# NHAMCS Scenario 3 — COVID-19 in the ED, and who gets admitted

_A new disease in a routine survey, and a prognostic model of admission. Level: third analysis. About two hours._

**Before you start.** NHAMCS is a national probability sample of US emergency department visits:
86,864 visits abstracted from the records of about 200 hospitals a year, 2018 to 2022. Every record
carries a weight, the number of US visits it stands for, and **the MIP does not use weights**: every
figure in this notebook describes *the sampled visits*, not US emergency care. CDC codes are kept:
questions are 1 = Yes, 2 = No; checkbox items 0 = not marked, 1 = marked; Sex is 1 = Female,
2 = Male; unknown is 98. One row is one visit, and the same person may appear twice.

**How this notebook works.** Every analysis is a *federated* call: the code selects variables,
defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a model). No
visit record ever reaches this notebook. Run the cells in order; each step says what to look for
and what result to expect.

**Learning goals.** Select years and build a cohort with filters; watch a new diagnosis appear
month by month; compare presentation and care between two groups of visits; describe the
predictors of hospital admission one at a time; and fit a multivariable logistic regression whose
odds ratios you can explain to a clinician.

**Clinical background.** The ICD-10-CM code U07.1 for COVID-19 was introduced on 1 April 2020.
NHAMCS coded up to five diagnoses per visit, so from that month COVID-19 visits can be identified;
the curators derived a COVID-19 diagnosis flag (U07.1 among the diagnoses). The decision to
**admit** a patient is the central disposition decision of the ED. Age, comorbidity, acuity at
arrival and the presenting problem all weigh on it. NHAMCS records 22 chronic conditions and their
total, the arrival mode, the triage level and the vital signs, which allows a simple admission
model. COVID-19 coding depends on testing and documentation practice, which changed fast in 2020;
the flag captures visits in which COVID-19 was among the recorded diagnoses, not all infected
patients. 2020 has fewer sampled visits than other years.

## 1. Connect to the MIP and open the NHAMCS pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("NHAMCS_ED")   # add version="v1" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
ed = dm.datasets.list()[0]              # one dataset: the five survey years together
display(dm)
display(ed)

In [ ]:
# The variable tree follows the visit: Survey administration, Arrival and visit timing, Patient,
# Expected source of payment, Triage and initial vital signs, Reason for visit, Injury, Diagnoses,
# Chronic conditions, Diagnostic services, Procedures, Medications, Vital signs after triage,
# Providers seen, Disposition, ED and hospital characteristics.
dm.tree()

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the datasets of the pathology
PCT, MIN = "{:.1f}", "{:.0f}"           # number formats of the tables: percentages, minutes


def analysis(variables, filters=None, drop_missing=False):
    """A Pipeline on the NHAMCS dataset with the given variables and cohort filter.
    Tests and models need drop_missing=True: the platform requires an explicit missing-values
    step and analyses complete cases only. describe() keeps it off so it can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=[ed], variables=variables), filters=filters,
                        handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def rows_for(result, variable):
    """The pooled 'describe' row of one variable (the platform adds one row per dataset as well)."""
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") == POOLED]


def numeric_summary(result, variables):
    """n, missing, mean, SD, quartiles and range of numerical variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "q1": d.get("q1"), "median": d.get("q2"),
                         "q3": d.get("q3"), "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan})
    return pd.DataFrame(rows)


def yes_percent(result, variable, level="Yes"):
    """Percentage of one level (default Yes) of a categorical variable in a describe result."""
    k = category_counts(result, variable)
    return float(k.loc[k["level"] == level, "percent"].sum()) if len(k) else np.nan


def by_level(numeric, group, levels=None, base_filter=None):
    """n, mean, median and quartiles of a numerical variable in each level of a categorical one
    (one describe per level). levels: the codes to use, by default every code of the variable."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({group.label: lab.get(c, c), "n": d.get("num_dtps"), "mean": d.get("mean"),
                         "q1": d.get("q1"), "median": d.get("q2"), "q3": d.get("q3")})
    return pd.DataFrame(rows)


def crosstab(exposure, outcome, base_filter=None, event="1", levels=None):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    rows = []
    for c in levels or lab_x:
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: lab_x.get(c, c), "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row["% " + lab_y.get(event, event)] = 100 * counts.get(event, 0) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if levels:
        f = f & F(exposure).isin(list(levels))
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def rates_by(group, outcomes, levels=None, base_filter=None, event="1"):
    """% of visits with each outcome (code `event`) in every level of `group`: one describe per
    level for the counts of all outcomes, one Chi-squared Test per outcome (in .attrs)."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f).describe()
        row = {group.label: lab.get(c, c)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    table = pd.DataFrame(rows)
    table.attrs["chi2"], table.attrs["p"] = {}, {}
    for y in outcomes:
        f = F(group).is_not_null() & F(y).is_not_null()
        if levels:
            f = f & F(group).isin(list(levels))
        if base_filter is not None:
            f = f & base_filter
        chi = analysis([group, y], f, drop_missing=True).chi_square_test(x=group, y=y)
        table.attrs["chi2"][y.label] = chi.raw.get("chi2")
        table.attrs["p"][y.label] = chi.raw.get("p_value")
    return table


def band_rates(outcomes, bands, base_filter=None, event="1"):
    """% of visits with each outcome in bands of a variable: bands maps a label to a filter."""
    rows = []
    for label, f in bands.items():
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f).describe()
        row = {"band": label}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: "{:.1f}"}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:.1f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(6, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def anova_summary(result):
    """One-way ANOVA result -> F, p, n, a table of the group means and SDs, and the Tukey comparisons."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    table, ci = raw.get("anova_table", {}), raw.get("ci_info", {})
    groups = pd.DataFrame({"mean": pd.Series(ci.get("means", {})), "sd": pd.Series(ci.get("sample_stds", {}))})
    return table.get("f_stat"), table.get("p_value"), table.get("n_obs"), groups, pd.DataFrame(raw.get("tuckey_test", []))


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    by_code = {v._code: v for v in variables}

    def pretty(term):
        term = str(term)
        if "[" in term and term.endswith("]"):
            base, level = term[:-1].split("[", 1)
            v = by_code.get(base)
            return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
        v = by_code.get(term)
        return f"{v.label} (per unit)" if v else term

    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(pretty)
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The variables of this scenario

| Code | Label in the MIP | Tree location (under NHAMCS_ED) | Role | Values |
|---|---|---|---|---|
| `covid_dx` | COVID-19 diagnosis | Diagnoses | exposure; outcome of Step 1; covariate; curation-derived | 0 No; 1 Yes |
| `YEAR` | Survey year | Survey administration | cohort filter; exposure of Step 1 | 2018 2018; 2019 2019; 2020 2020; 2021 2021; 2022 2022 |
| `VMONTH` | Month of visit | Survey administration | exposure of Step 1 | 12 codes, listed in the tree |
| `admitted` | Admitted to hospital | Disposition | outcome; curation-derived | 0 No; 1 Yes |
| `AGE` | Age | Patient | predictor, continuous | integer, 0-94 years |
| `AGER` | Age group | Patient | predictor, grouped | 1 Under 15 years; 2 15-24 years; 3 25-44 years; 4 45-64 years; 5 65-74 years; 6 75 years and over |
| `TOTCHRON` | Number of chronic conditions | Chronic conditions | predictor | integer, 0-22 |
| `ARREMS` | Arrival by ambulance | Arrival and visit timing | predictor; cohort filter 1, 2 | 1 Yes; 2 No; 98 Unknown |
| `IMMEDR` | Triage level | Triage and initial vital signs | predictor; filter 1 to 5 | 0 No triage reported, ESA conducts nursing triage; 1 Immediate; 2 Emergent; 3 Urgent; 4 Semi-urgent; 5 Nonurgent; 7 Visit in an emergency service area that does not conduct nursing triage; 98 Unknown |
| `fever` | Fever at triage | Triage and initial vital signs | predictor; curation-derived | 0 No; 1 Yes |
| `rfv1_group` | Reason for visit 1, group | Reason for visit | presentation of COVID-19 visits; curation-derived | 17 codes, listed in the tree |
| `XRAY` | X-ray | Diagnostic services / Imaging | care of COVID-19 visits | 0 No; 1 Yes |
| `CTCHEST` | CT chest | Diagnostic services / Imaging | care of COVID-19 visits | 0 No; 1 Yes |
| `med_glucocorticoid` | Systemic glucocorticoid | Medications / Drug classes | care of COVID-19 visits; curation-derived | 0 No; 1 Yes |
| `SEX` | Sex | Patient | covariate | 1 Female; 2 Male |
| `TOTDIAG` | Number of diagnostic services | Diagnostic services | consequence of the work-up (Step 4) | integer, 0-30 |
| `NUMGIV` | Medications given in ED | Medications | consequence of the work-up (Step 4) | integer, 0-30 |
| `disposition` | Disposition summary | Disposition | extension; curation-derived | 1 Treated and released (no other disposition marked); 2 Admitted to this hospital (directly or via observation); 3 Observation unit, then discharged; 4 Transferred (other hospital, psychiatric hospital, nursing home); 5 Left without being seen, before treatment complete, or against medical advice; 6 Died (dead on arrival or died in ED); 7 Other disposition only |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `WAITTIME`) is what the
csv column is called; you will see it in some result payloads.

In [ ]:
covid     = dm.variables["COVID-19 diagnosis"]
year      = dm.variables["Survey year"]
month     = dm.variables["Month of visit"]
admitted  = dm.variables["Admitted to hospital"]
age       = dm.variables["Age"]
ager      = dm.variables["Age group"]
chronic   = dm.variables["Number of chronic conditions"]
ambulance = dm.variables["Arrival by ambulance"]
triage    = dm.variables["Triage level"]
fever     = dm.variables["Fever at triage"]
reason    = dm.variables["Reason for visit 1, group"]
xray      = dm.variables["X-ray"]
ct_chest  = dm.variables["CT chest"]
steroid   = dm.variables["Systemic glucocorticoid"]
sex       = dm.variables["Sex"]
totdiag   = dm.variables["Number of diagnostic services"]
numgiv    = dm.variables["Medications given in ED"]
display(mip.to_frame([covid, year, month, admitted, age, ager, chronic, ambulance, triage, fever,
                      reason, xray, ct_chest, steroid, sex, totdiag, numgiv]))

## Step 1. COVID-19 appears

The cross-tabulation of the COVID-19 flag by survey year on all visits, then by month of visit
within 2020.

*Expected:* 0% in 2018 and 2019 (the code did not exist), 2.0% of 2020 visits (298), 3.9% of 2021
(628), 4.6% of 2022 (731). Within 2020: a single coded visit in January and in February, none in
March, 2.1% in April, 1 to 2% through the summer, 4.7% in November and 8.4% in December, the winter
wave. A routine survey records history as it happens.

In [ ]:
by_year = crosstab(year, covid)
show_crosstab(by_year, "Visits with a COVID-19 diagnosis, by survey year")

in_2020 = F(year) == "2020"
by_month = crosstab(month, covid, base_filter=in_2020)
display(by_month[[month.label, "n", "Yes", "% Yes"]].style.format({"% Yes": PCT}).hide(axis="index"))
ax = by_month.set_index(month.label)["% Yes"].plot(kind="bar", color="#C44E52", figsize=(8, 3), rot=45)
ax.set_ylabel("% of visits"); ax.set_xlabel(""); ax.set_title("COVID-19 diagnoses in the sampled visits of 2020, by month")
plt.tight_layout()
plt.show()

## Step 2. Who came with COVID-19?

The pandemic years, 2020 to 2022 (47,092 visits). The COVID-19 flag by age group, then one
`describe()` of the presentation and care variables among COVID-19 visits and one among the other
visits, and the *Chi-squared Test* of admission by COVID-19 status.

*Expected:* COVID-19 was coded in 2.3 to 2.6% of visits under 25 years and 4.0 to 4.8% of visits at
45 and over. Among the 1,657 COVID-19 visits (1,653 with a recorded disposition), the first reason
for visit was a respiratory symptom in 35% and a general symptom (fever, malaise) in 24%; fever at
triage in 15% versus 3% of other
visits; chest X-ray in 54% versus 34%; chest CT in 7% versus 3%; systemic glucocorticoid in 13%
versus 6%; admitted 21.6% versus 12.7% (chi-square 109, p < 0.001). A COVID-19 visit was almost
twice as likely to end in admission as any other visit.

In [ ]:
pandemic = F(year).isin(["2020", "2021", "2022"])
show_crosstab(crosstab(ager, covid, base_filter=pandemic), "COVID-19 diagnosis by age group, 2020 to 2022")

profile = [reason, fever, xray, ct_chest, steroid, admitted]
with_covid = analysis(profile, pandemic & (F(covid) == "1")).describe()
without = analysis(profile, pandemic & (F(covid) == "0")).describe()
n_covid = sum(category_counts(with_covid, admitted)["count"])
compare = pd.DataFrame([{"": v.label, f"COVID-19 visits (n = {n_covid:,})": yes_percent(with_covid, v),
                         "other visits": yes_percent(without, v)} for v in profile[1:]]).set_index("")
display(compare.style.format(PCT))
reasons = category_counts(with_covid, reason).sort_values("percent", ascending=False).head(5)
display(reasons[["level", "count", "percent"]].rename(columns={"level": "first reason for visit, COVID-19 visits"})
        .style.format({"percent": PCT}).hide(axis="index"))

chi = analysis([covid, admitted], pandemic, drop_missing=True).chi_square_test(x=covid, y=admitted)
print(f"Admission by COVID-19 status: chi2 = {chi.raw['chi2']:.0f}, p = {fmt_p(chi.raw['p_value'])}")
ax = compare.plot(kind="barh", figsize=(7, 3.2), color=["#C44E52", "#4C72B0"])
ax.invert_yaxis(); ax.set_xlabel("% of visits"); ax.set_title("COVID-19 visits against the others, 2020 to 2022")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Step 3. Predictors of admission, one at a time

Still 2020 to 2022, all visits. Admission cross-tabulated against age group, arrival by ambulance,
triage level (levels 1 to 5) and fever at triage; for the number of chronic conditions, one
`describe()` per band (0, 1, 2, 3 or more).

*Expected percentages admitted:*

| Factor | Levels |
|---|---|
| Age group | under 15: 4.4%; 15-24: 5.5%; 25-44: 7.8%; 45-64: 16.0%; 65-74: 25.3%; 75+: 32.9% |
| Chronic conditions | 0: 4.8%; 1: 11.3%; 2: 16.5%; 3 or more: 30.7% |
| Arrival by ambulance | yes 30.1%; no 8.9% |
| Triage level | 1: 40%; 2: 34%; 3: 13%; 4: 1.7%; 5: 4.7% |
| Fever at triage | yes 19.2%; no 12.5% |

All p < 0.001. Age and comorbidity show clean gradients; ambulance arrival triples the admission
rate; triage level 1 and 2 patients are admitted a third of the time.

In [ ]:
factors = []
for exposure, levels in ((ager, None), (ambulance, ["1", "2"]), (triage, ["1", "2", "3", "4", "5"]), (fever, None)):
    t = crosstab(exposure, admitted, base_filter=pandemic, levels=levels)
    print(f"{exposure.label}: chi2 = {t.attrs['chi2']:.0f}, p = {fmt_p(t.attrs['p'])}")
    for _, r in t.iterrows():
        factors.append({"factor": exposure.label, "level": r[exposure.label], "n": r["n"], "% admitted": r["% Yes"]})
bands = {"0": F(chronic) == 0, "1": F(chronic) == 1, "2": F(chronic) == 2, "3 or more": F(chronic) >= 3}
for _, r in band_rates([admitted], bands, base_filter=pandemic).iterrows():
    factors.append({"factor": chronic.label, "level": r["band"], "n": r["n"], "% admitted": r[admitted.label]})
factors = pd.DataFrame(factors)
display(factors.style.format({"% admitted": PCT}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 6))
labels = factors["factor"] + ": " + factors["level"].astype(str)
ax.barh(labels[::-1], factors["% admitted"][::-1], color="#4C72B0")
ax.set_xlabel("% admitted"); ax.set_title("Admission by each factor, 2020 to 2022")
plt.tight_layout()
plt.show()

## Step 4. Continuous comparisons

An *Independent t-test* of age, number of chronic conditions, number of diagnostic services and
medications given in the ED between admitted and discharged visits, with the two group means from
one `describe()` per group.

*Expected:* admitted patients are older (57 versus 38 years), have more chronic conditions (2.8
versus 1.1), more diagnostic services (7.5 versus 3.2) and more drugs given (4.0 versus 1.5); all
t above 50, p < 0.001. The last two are consequences of the work-up, not predictors available at
arrival: keep that distinction when building a model.

In [ ]:
measures = [age, chronic, totdiag, numgiv]
means = {}
for label, c in (("admitted", "1"), ("not admitted", "0")):
    res = analysis(measures, pandemic & (F(admitted) == c)).describe()
    means[label] = numeric_summary(res, measures).set_index("variable")["mean"]
rows = []
for v in measures:
    t = analysis([v, admitted], pandemic, drop_missing=True).t_test(variable=v, group_by=admitted, group_a="1", group_b="0")
    s = ttest_summary(t)
    rows.append({"measure": v.label, "mean, admitted": means["admitted"][v.label], "mean, not admitted": means["not admitted"][v.label],
                 "difference": s["mean_diff"], "t": s["t_stat"], "p": s["p"], "n": s["n_obs"]})
display(pd.DataFrame(rows).style.format({"mean, admitted": PCT, "mean, not admitted": PCT, "difference": PCT, "t": PCT, "p": fmt_p}).hide(axis="index"))

## Step 5. A multivariable admission model

Visits of 2020 to 2022 with a known arrival mode and a known number of chronic conditions, and a
*Logistic Regression* of admission on age, number of chronic conditions, arrival by ambulance,
COVID-19 diagnosis and sex. The platform compares each category with its first code: for arrival
by ambulance that is 1 = Yes, so its line reads "No" with an odds ratio below 1, the inverse of
the ambulance effect; the other references are 0 = no COVID-19 diagnosis and 1 = female.

*Expected (44,592 visits: the platform also leaves out the 206 visits whose disposition is
blank):*

| Term | Odds ratio | 95% CI |
|---|---|---|
| Age, per year | 1.019 | 1.017 to 1.020 |
| Chronic conditions, per condition | 1.29 | 1.27 to 1.31 |
| Arrival by ambulance = No (as the platform shows it) | 0.36 | 0.34 to 0.38 |
| Arrival by ambulance (versus none), inverted: 1 / 0.36 | 2.78 | 2.61 to 2.96 |
| COVID-19 diagnosis | 1.81 | 1.58 to 2.08 |
| Male | 1.13 | 1.07 to 1.20 |

Read it: for the same age, comorbidity, sex and arrival mode, a COVID-19 diagnosis still raised
the odds of admission by 80%; each chronic condition adds 29%; ambulance arrival nearly triples the
odds, partly because ambulances bring sicker patients and partly because paramedics select those
who need hospital care. Age at 1.9% per year means a 70-year-old has about 3.4 times the odds of a
5-year-old, at the same comorbidity count.

In [ ]:
cohort = pandemic & F(ambulance).isin(["1", "2"]) & F(chronic).is_not_null()
covariates = [age, chronic, ambulance, covid, sex]
model = analysis([admitted] + covariates, cohort, drop_missing=True).logistic_regression(
    x=covariates, y=admitted, positive_class="1")
print(f"{model.raw['summary']['n_obs']:,} visits in the model")
ors = odds_ratio_table(model, covariates)
show_odds_ratios(ors, "Odds of admission, sampled visits of 2020 to 2022")

no_ambulance = ors.set_index("term").loc["Arrival by ambulance = No"]
print(f"Ambulance versus no ambulance, the inverse of the platform's term: "
      f"{1 / no_ambulance['odds_ratio']:.2f} (95% CI {1 / no_ambulance['ci_high']:.2f} to {1 / no_ambulance['ci_low']:.2f})")

## Step 6. State the result

A model answer:

> Among 47,092 sampled US ED visits in 2020 to 2022, COVID-19 was among the diagnoses in 3.5% (from
> 2% in 2020 to 4.6% in 2022), appearing in April 2020 and peaking in December 2020. COVID-19
> visits presented more often with respiratory or general symptoms and fever, received more chest
> imaging and glucocorticoids, and were admitted in 22% of cases versus 13% of other visits.
> Admission was driven by age, number of chronic conditions, arrival by ambulance and acuity;
> after adjustment, a COVID-19 diagnosis multiplied the odds of admission by 1.8 (95% CI 1.6 to
> 2.1). These are unweighted sample estimates and the COVID-19 flag depends on coding practice.

**Questions for discussion.** The COVID-19 flag is 0 for every 2018 and 2019 visit: why is a
comparison of "COVID versus non-COVID" across all five years misleading? Ambulance arrival has the
largest odds ratio: is it a cause of admission, a marker of severity, or a selection effect? The
number of diagnostic services and of medications given differ enormously between admitted and
discharged patients: why were they left out of the model? The model uses one visit per row, but
a patient may appear more than once and visits cluster within hospitals: what does this do to the
confidence intervals?

**Optional extension.** Compare the pandemic year with the year before: filter 2019, then 2020,
and compare arrival by ambulance, triage level, admission and the disposition summary. Visits
fell by a quarter in the 2020 sample; did the mix of patients change? Then repeat for the first
reason for visit and discuss who stayed away from the ED in 2020.

In [ ]:
dispo = dm.variables["Disposition summary"]
mix = [ambulance, triage, admitted, dispo, reason]
by_year = {y: analysis(mix, F(year) == y).describe() for y in ("2019", "2020")}
rows = []
for v in mix:
    k = {y: category_counts(by_year[y], v).set_index("level")["percent"] for y in by_year}
    for level in k["2019"].index:
        rows.append({"variable": v.label, "level": level, "2019": k["2019"].get(level), "2020": k["2020"].get(level)})
change = pd.DataFrame(rows)
change["difference"] = change["2020"] - change["2019"]
display(change.style.format({"2019": PCT, "2020": PCT, "difference": "{:+.1f}"}).hide(axis="index"))